In [1]:
import spacy
import pandas as pd
import random
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import kagglehub
from kagglehub import KaggleDatasetAdapter

c:\Users\prana\Desktop\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download("abhinavwalia95/entity-annotated-corpus")

In [3]:
print("Path to dataset files:", path)

Path to dataset files: C:\Users\prana\.cache\kagglehub\datasets\abhinavwalia95\entity-annotated-corpus\versions\4


In [4]:
df = pd.read_csv(r"C:\Users\prana\.cache\kagglehub\datasets\abhinavwalia95\entity-annotated-corpus\versions\4\ner_dataset.csv", encoding="latin1")

In [5]:
df["Sentence #"] = df["Sentence #"].ffill()

In [6]:
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2_000_000  # safety
print("spaCy NER model loaded")

spaCy NER model loaded


In [7]:
sentences = []
labels = []

current_sentence = []
current_labels = []

prev_sentence_id = None

for _, row in df.iterrows():
    sentence_id = row["Sentence #"]
    word = row["Word"]
    tag = row["Tag"]

    if sentence_id != prev_sentence_id:
        if current_sentence:
            sentences.append(current_sentence)
            labels.append(current_labels)
        current_sentence = [word]
        current_labels = [tag]
        prev_sentence_id = sentence_id
    else:
        current_sentence.append(word)
        current_labels.append(tag)

# add last sentence
if current_sentence:
    sentences.append(current_sentence)
    labels.append(current_labels)

print("Total sentences:", len(sentences))


Total sentences: 47959


In [8]:
def extract_entities(tokens, tags):
    entities = []
    start = None
    label = None

    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start is not None:
                entities.append((start, i, label))
            start = i
            label = tag[2:]

        elif tag.startswith("I-") and label == tag[2:]:
            continue
        else:
            if start is not None:
                entities.append((start, i, label))
                start = None
                label = None

    if start is not None:
        entities.append((start, len(tokens), label))

    return entities


In [9]:
MAX_CHARS = 500

y_true = []
y_pred = []

for tokens, tags in zip(sentences[:300], labels[:300]):  # limit for speed
    text = " ".join(tokens)

    if len(text) > MAX_CHARS:
        continue

    doc = nlp(text)

    true_entities = extract_entities(tokens, tags)
    pred_entities = [(ent.start, ent.end, ent.label_) for ent in doc.ents]

    for ent in true_entities:
        y_true.append(ent[2])
        y_pred.append(ent[2] if ent in pred_entities else "O")

    for ent in pred_entities:
        if ent not in true_entities:
            y_true.append("O")
            y_pred.append(ent[2])


In [10]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)

accuracy = accuracy_score(y_true, y_pred)

print("Accuracy :", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall   :", round(recall, 3))
print("F1 Score :", round(f1, 3))

Accuracy : 0.0
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
